In [ ]:
!pip uninstall -y torch torchvision torchaudio

In [ ]:
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.3.1+cu118
11.8
Tesla P100-PCIE-16GB


In [14]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import nibabel as nib
import cv2
import pandas as pd
from scipy.ndimage import label as scipy_label
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────
# VIEW CONFIG  (label counts come directly from the challenge)
# ─────────────────────────────────────────────────────────────
VIEW_CONFIG = {
    #  view       num_fg  total(+bg)  lv_cavity_label
    "SAX_TR": {"num_classes": 4, "lv_label": 2},   # 0=bg,1=LV-myo,2=LV-cav,3=RV-cav
    "2CH_TR": {"num_classes": 3, "lv_label": 1},   # 0=bg,1=LV-cav,2=LV-myo
    "4CH_TR": {"num_classes": 6, "lv_label": 1},   # 0=bg,1=LV-cav,2=LV-myo,3=RV-cav,4=RA,5=LA
}


# ─────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4):
        super().__init__()
        self.d1, self.p1 = DoubleConv(in_channels, 64),  nn.MaxPool2d(2)
        self.d2, self.p2 = DoubleConv(64, 128),           nn.MaxPool2d(2)
        self.d3, self.p3 = DoubleConv(128, 256),          nn.MaxPool2d(2)
        self.d4, self.p4 = DoubleConv(256, 512),          nn.MaxPool2d(2)
        self.bn          = DoubleConv(512, 1024)

        self.u4 = nn.ConvTranspose2d(1024, 512, 2, stride=2); self.c4 = DoubleConv(1024, 512)
        self.u3 = nn.ConvTranspose2d(512,  256, 2, stride=2); self.c3 = DoubleConv(512,  256)
        self.u2 = nn.ConvTranspose2d(256,  128, 2, stride=2); self.c2 = DoubleConv(256,  128)
        self.u1 = nn.ConvTranspose2d(128,   64, 2, stride=2); self.c1 = DoubleConv(128,   64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        d1 = self.d1(x);          d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2)); d4 = self.d4(self.p3(d3))
        bn = self.bn(self.p4(d4))
        x = self.c4(torch.cat([self.u4(bn), d4], 1))
        x = self.c3(torch.cat([self.u3(x),  d3], 1))
        x = self.c2(torch.cat([self.u2(x),  d2], 1))
        x = self.c1(torch.cat([self.u1(x),  d1], 1))
        return self.final(x)


# ─────────────────────────────────────────────────────────────
# LOSS
# ─────────────────────────────────────────────────────────────
class DiceCELoss(nn.Module):
    def __init__(self, num_classes, ce_w=0.5, dice_w=0.5, smooth=1e-5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ce_w = ce_w; self.dice_w = dice_w

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        probs   = F.softmax(logits, dim=1)
        t_oh    = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dice = 0.0
        for c in range(1, self.num_classes):
            p = probs[:, c]; t = t_oh[:, c]
            dice += 1 - (2*(p*t).sum() + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return self.ce_w * ce_loss + self.dice_w * dice / (self.num_classes - 1)


# ─────────────────────────────────────────────────────────────
# DATASET  — loads real LVEF from dataset.xlsx for SAX
# ─────────────────────────────────────────────────────────────
class CineMRIDataset(Dataset):
    def __init__(self, root_dir, view="SAX_TR", img_size=256, augment=False):
        self.img_dir  = os.path.join(root_dir, view, "image")
        self.mask_dir = os.path.join(root_dir, view, "anno")
        self.img_size = img_size
        self.augment  = augment
        self.view     = view

        # ── Load real LVEF from Excel (SAX only) ──────────────
        # Key fix: EF ground truth lives in dataset.xlsx, not in pixel counts
        self.lvef_map = {}   # {filename_stem: lvef_value}
        xlsx_path = os.path.join(root_dir, "dataset.xlsx")
        sheet_map = {"SAX_TR": "SAX", "2CH_TR": "2CH", "4CH_TR": "4CH"}
        if os.path.exists(xlsx_path) and view in sheet_map:
            try:
                df = pd.read_excel(xlsx_path, sheet_name=sheet_map[view])
                if "LVEF" in df.columns and "image_path" in df.columns:
                    for _, row in df.iterrows():
                        stem = os.path.splitext(os.path.basename(str(row["image_path"])))[0]
                        stem = stem.replace(".nii", "")   # handle .nii.gz
                        self.lvef_map[stem] = float(row["LVEF"])
            except Exception as e:
                print(f"[WARN] Could not load LVEF from xlsx: {e}")

        # ── Build sample list ──────────────────────────────────
        self.files   = sorted(os.listdir(self.img_dir))
        self.samples = []   # (img_path, mask_path, t, s_or_None, lvef_or_nan)

        for file in self.files:
            img_path  = os.path.join(self.img_dir,  file)
            mask_path = os.path.join(self.mask_dir, file)
            stem      = file.replace(".nii.gz", "").replace(".nii", "")
            lvef      = self.lvef_map.get(stem, float("nan"))

            img = nib.load(img_path).get_fdata()
            if img.ndim == 4:
                H, W, T, S = img.shape
                for s in range(S):
                    for t in range(T):
                        self.samples.append((img_path, mask_path, t, s, lvef))
            elif img.ndim == 3:
                H, W, T = img.shape
                for t in range(T):
                    self.samples.append((img_path, mask_path, t, None, lvef))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, t, s, lvef = self.samples[idx]

        img  = nib.load(img_path).get_fdata()
        mask = nib.load(mask_path).get_fdata()

        if img.ndim == 4:
            img  = img[:, :, t, s]
            mask = mask[:, :, t, s]
        else:
            img  = img[:, :, t]
            mask = mask[:, :, t]

        img = (img - img.mean()) / (img.std() + 1e-8)
        img  = cv2.resize(img,  (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size),
                          interpolation=cv2.INTER_NEAREST)

        if self.augment:
            img, mask = self._augment(img, mask)

        img  = torch.tensor(img,  dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.long)
        return img, mask, torch.tensor(lvef, dtype=torch.float32)

    def _augment(self, img, mask):
        if np.random.rand() > 0.5:
            img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
        if np.random.rand() > 0.5:
            img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
        k = np.random.randint(0, 4)
        img, mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy()
        alpha = np.random.uniform(0.8, 1.2)
        beta  = np.random.uniform(-0.1, 0.1)
        img   = np.clip(alpha * img + beta, img.min(), img.max())
        return img, mask


# ─────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────
def dice_score(preds, targets, num_classes):
    preds = torch.argmax(preds, dim=1)
    dice  = 0.0
    for c in range(1, num_classes):
        p = (preds   == c).float()
        t = (targets == c).float()
        dice += (2*(p*t).sum() + 1e-5) / (p.sum() + t.sum() + 1e-5)
    return dice / (num_classes - 1)


def hausdorff95_asd(pred_np, gt_np, num_classes):
    """2D per-sample HD95 and ASD, averaged over foreground classes."""
    hds, asds = [], []
    for c in range(1, num_classes):
        p = (pred_np == c); t = (gt_np == c)
        if p.sum() == 0 or t.sum() == 0:
            continue
        # Surface points via erosion
        from scipy.ndimage import binary_erosion
        p_surf = np.column_stack(np.where(p & ~binary_erosion(p)))
        t_surf = np.column_stack(np.where(t & ~binary_erosion(t)))
        if len(p_surf) == 0 or len(t_surf) == 0:
            continue
        D = cdist(p_surf, t_surf)
        d_pt = D.min(axis=1); d_tp = D.min(axis=0)
        hds.append(max(np.percentile(d_pt, 95), np.percentile(d_tp, 95)))
        asds.append((d_pt.mean() + d_tp.mean()) / 2.0)
    return (np.mean(hds), np.mean(asds)) if hds else (float("nan"), float("nan"))


def keep_largest_component(pred, num_classes):
    clean = np.zeros_like(pred)
    for c in range(1, num_classes):
        binary = (pred == c)
        labeled, n = scipy_label(binary)
        if n == 0: continue
        largest = np.argmax([(labeled == i).sum() for i in range(1, n+1)]) + 1
        clean[labeled == largest] = c
    return clean


def estimate_lv_pixels(pred_np, lv_label):
    """Count LV cavity pixels — used to estimate ED/ES volumes for EF."""
    return float((pred_np == lv_label).sum())


# ─────────────────────────────────────────────────────────────
# TRAINING  (per-view, single script)
# ─────────────────────────────────────────────────────────────
def train_view(root_dir, view="SAX_TR", num_epochs=15, batch_size=32, lr=1e-4, img_size=256):
    cfg         = VIEW_CONFIG[view]
    num_classes = cfg["num_classes"]
    lv_label    = cfg["lv_label"]

    print(f"\n{'='*60}")
    print(f"  Training on view: {view}  |  num_classes: {num_classes}  |  LV label: {lv_label}")
    print(f"{'='*60}")

    # ── Datasets ──────────────────────────────────────────────
    train_ds = CineMRIDataset(root_dir, view=view, img_size=img_size, augment=True)
    val_ds   = CineMRIDataset(root_dir, view=view, img_size=img_size, augment=False)

    indices = list(range(len(train_ds)))
    np.random.shuffle(indices)
    split = int(0.8 * len(indices))

    train_loader = DataLoader(Subset(train_ds, indices[:split]),
                              batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(Subset(val_ds,   indices[split:]),
                              batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = UNet(in_channels=1, out_channels=num_classes)

    # Use multiple GPUs if available
    if torch.cuda.device_count() > 1:
      print(f"Using {torch.cuda.device_count()} GPUs")
      model = nn.DataParallel(model)

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = DiceCELoss(num_classes=num_classes)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_dice = 0.0

    for epoch in range(num_epochs):
        # ── Train ───────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for imgs, masks, _ in tqdm(train_loader, desc=f"[{view}] Epoch {epoch+1}/{num_epochs}"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss  = criterion(preds, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        # ── Validate ─────────────────────────────────────────
        model.eval()
        val_loss = 0.0; val_dice = 0.0
        val_hd   = 0.0; val_asd  = 0.0; hd_n = 0

        # For EF PCC: collect (predicted_lv_pixels, true_lvef)
        pred_lv_vols, gt_lvefs = [], []

        with torch.no_grad():
            for imgs, masks, lvefs in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs)

                val_loss += criterion(preds, masks).item()
                val_dice += dice_score(preds, masks, num_classes).item()

                pred_np = torch.argmax(preds, 1).cpu().numpy()
                mask_np = masks.cpu().numpy()

                for b in range(pred_np.shape[0]):
                    p_clean = keep_largest_component(pred_np[b], num_classes)
                    hd, asd = hausdorff95_asd(p_clean, mask_np[b], num_classes)
                    if not np.isnan(hd):
                        val_hd += hd; val_asd += asd; hd_n += 1

                    # EF PCC: only meaningful on SAX with real LVEF labels
                    lv_px = estimate_lv_pixels(p_clean, lv_label)
                    lvef_val = lvefs[b].item()
                    if not np.isnan(lvef_val):
                        pred_lv_vols.append(lv_px)
                        gt_lvefs.append(lvef_val)

        n = len(val_loader)
        avg_dice = val_dice / n
        avg_hd   = val_hd / hd_n if hd_n > 0 else float("nan")
        avg_asd  = val_asd / hd_n if hd_n > 0 else float("nan")

        # EF PCC needs ≥2 paired samples with real LVEF values
        if view == "SAX_TR" and len(gt_lvefs) >= 2:
            pcc, _ = pearsonr(pred_lv_vols, gt_lvefs)
        else:
            pcc = float("nan")   # Not applicable for 2CH/4CH

        print(f"\n[{view}] Epoch {epoch+1:02d}/{num_epochs}")
        print(f"  Train Loss : {train_loss/n:.4f}")
        print(f"  Val   Loss : {val_loss/n:.4f}")
        print(f"  Val Dice   : {avg_dice:.4f}  ↑")
        print(f"  Val HD95   : {avg_hd:.2f} mm  ↓")
        print(f"  Val ASD    : {avg_asd:.2f} mm  ↓")
        print(f"  EF PCC     : {pcc:.4f}  ↑" if not np.isnan(pcc) else "  EF PCC     : N/A (no LVEF labels or non-SAX view)")

        if avg_dice > best_dice:
            best_dice = avg_dice
            torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict(),
             f"unet_{view}_best.pth")
            print(f"  ✓ Best model saved")
    torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel)
           else model.state_dict(),
           f"unet_{view}_best.pth")
    return model


# ─────────────────────────────────────────────────────────────
# ENTRY POINT — train all three views
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    ROOT = "/kaggle/input/datasets/youssefsharabas/cinemultii"

    for view in ["SAX_TR", "2CH_TR", "4CH_TR"]:
        train_view(ROOT, view=view, num_epochs=15, batch_size=32)


  Training on view: SAX_TR  |  num_classes: 4  |  LV label: 2
Using 2 GPUs


[SAX_TR] Epoch 1/15: 100%|██████████| 741/741 [10:55<00:00,  1.13it/s]



[SAX_TR] Epoch 01/15
  Train Loss : 1.4360
  Val   Loss : 0.1736
  Val Dice   : 0.8702  ↑
  Val HD95   : 9.25 mm  ↓
  Val ASD    : 3.12 mm  ↓
  EF PCC     : N/A (no LVEF labels or non-SAX view)
  ✓ Best model saved


[SAX_TR] Epoch 2/15: 100%|██████████| 741/741 [10:55<00:00,  1.13it/s]



[SAX_TR] Epoch 02/15
  Train Loss : 0.5131
  Val   Loss : 0.1196
  Val Dice   : 0.8735  ↑
  Val HD95   : 8.60 mm  ↓
  Val ASD    : 2.83 mm  ↓
  EF PCC     : N/A (no LVEF labels or non-SAX view)
  ✓ Best model saved


...


[4CH_TR] Epoch 11/15
  Train Loss : 0.3558
  Val   Loss : 0.0883
  Val Dice   : 0.9248  ↑
  Val HD95   : 4.35 mm  ↓
  Val ASD    : 1.54 mm  ↓
  EF PCC     : N/A (no LVEF labels or non-SAX view)
  ✓ Best model saved


[4CH_TR] Epoch 12/15: 100%|██████████| 208/208 [03:07<00:00,  1.11it/s]



[4CH_TR] Epoch 12/15
  Train Loss : 0.3460
  Val   Loss : 0.0863
  Val Dice   : 0.9261  ↑
  Val HD95   : 4.31 mm  ↓
  Val ASD    : 1.53 mm  ↓
  EF PCC     : N/A (no LVEF labels or non-SAX view)
  ✓ Best model saved


[4CH_TR] Epoch 15/15: 100%|██████████| 208/208 [03:07<00:00,  1.11it/s]



[4CH_TR] Epoch 15/15
  Train Loss : 0.3267
  Val   Loss : 0.0828
  Val Dice   : 0.9286  ↑
  Val HD95   : 4.11 mm  ↓
  Val ASD    : 1.47 mm  ↓
  EF PCC     : N/A (no LVEF labels or non-SAX view)
  ✓ Best model saved
